# Bounded Stack Data Structure, Invariants, and Applications

**Session:** 2026-11-03 · OOP Practice Lab  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

A **Stack** is a linear abstract data type (ADT) operating on the **Last-In, First-Out (LIFO)** principle. Under a bounded capacity constraint:
- Maximum size $C$ is fixed at initialization time.
- Attempting to `push` onto a full stack ($|S| = C$) triggers a **Stack Overflow** condition (`OverflowError`).
- Attempting to `pop` or inspect `top` on an empty stack ($|S| = 0$) triggers a **Stack Underflow** condition (`IndexError`).

In this notebook, we implement and test:
1. Bounded `Stack` class conforming to lecture specifications (`capacity`, `isEmpty`, `isFull`, `push`, `pop`, `top`).
2. Robust exception handling protecting internal invariants.
3. Execution trace matching the 14-step lecture simulation.
4. Classical applications:
   - Balanced brackets validator.
   - Postfix (Reverse Polish Notation) arithmetic evaluator.


## 1. Bounded Stack Implementation

We implement the `Stack` class with defensive type validation and clean error semantics.


In [ ]:
from typing import Any


class Stack:
    """Bounded LIFO Stack data structure with capacity limits."""

    def __init__(self, capacity: int) -> None:
        if not isinstance(capacity, int) or capacity <= 0:
            raise ValueError(f"Capacity must be a strictly positive integer, got {capacity}")
        self.__capacity: int = int(capacity)
        self.__stack: list[Any] = []

    def get_capacity(self) -> int:
        """Return the maximum capacity of the stack."""
        return self.__capacity

    def is_empty(self) -> bool:
        """Check if the stack is currently empty."""
        return len(self.__stack) == 0

    def is_full(self) -> bool:
        """Check if the stack has reached maximum capacity."""
        return len(self.__stack) == self.__capacity

    def push(self, value: Any) -> None:
        """Push a value onto the top of the stack.

        Raises:
            OverflowError: If the stack has reached capacity.
        """
        if self.is_full():
            raise OverflowError(
                f"Stack Overflow: cannot push '{value}', capacity {self.__capacity} reached."
            )
        self.__stack.append(value)

    def pop(self) -> Any:
        """Remove and return the topmost element from the stack.

        Raises:
            IndexError: If the stack is empty (Underflow).
        """
        if self.is_empty():
            raise IndexError("Stack Underflow: cannot pop from an empty stack.")
        return self.__stack.pop()

    def top(self) -> Any:
        """Return the topmost element without removing it.

        Raises:
            IndexError: If the stack is empty.
        """
        if self.is_empty():
            raise IndexError("Stack Underflow: cannot inspect top of an empty stack.")
        return self.__stack[-1]

    def __len__(self) -> int:
        return len(self.__stack)

    def __repr__(self) -> str:
        return f"Stack(capacity={self.__capacity}, size={len(self.__stack)}, items={self.__stack})"


# Verify baseline functionality
s = Stack(capacity=4)
assert s.is_empty() is True
assert s.is_full() is False
s.push(10)
assert s.top() == 10
assert len(s) == 1
assert s.pop() == 10
assert s.is_empty() is True
print("Baseline Stack operations verified successfully.")

## 2. Lecture Trace Simulation

We reconstruct the exact sequence of operations presented in the lecture slide deck:
1. Initialize Stack with capacity 4.
2. `push(31)`
3. `push(12)`
4. `push(10)`
5. `push(20)` -> Stack is now FULL.
6. Verify `is_full() == True`.
7. `push(99)` -> Triggers `OverflowError`.
8. `pop()` -> Returns 20.
9. `pop()` -> Returns 10.
10. `top()` -> Returns 12 without mutating stack.
11. `pop()` -> Returns 12.
12. `pop()` -> Returns 31.
13. Verify `is_empty() == True`.
14. `pop()` -> Triggers `IndexError` (Underflow).


In [ ]:
stack_trace = Stack(capacity=4)

# Steps 1-5: Fill to capacity
for val in [31, 12, 10, 20]:
    stack_trace.push(val)

assert stack_trace.is_full() is True
assert stack_trace.top() == 20

# Step 7: Test Overflow
try:
    stack_trace.push(99)
    raise AssertionError("Failed to catch Stack Overflow.")
except OverflowError as e:
    print("Caught expected Overflow:", e)

# Steps 8-10: Pops and Top
assert stack_trace.pop() == 20
assert stack_trace.pop() == 10
assert stack_trace.top() == 12  # Top does not remove 12
assert len(stack_trace) == 2

# Steps 11-13: Drain to empty
assert stack_trace.pop() == 12
assert stack_trace.pop() == 31
assert stack_trace.is_empty() is True

# Step 14: Test Underflow
try:
    stack_trace.pop()
    raise AssertionError("Failed to catch Stack Underflow.")
except IndexError as e:
    print("Caught expected Underflow:", e)

print("Lecture slide trace simulation completed with 100% fidelity.")

## 3. Practical Application: Balanced Bracket Validation

An iconic application of the LIFO stack is syntactic validation of nested delimiters (e.g., in compilers and JSON parsers).


In [ ]:
def is_balanced_delimiters(expression: str) -> bool:
    """Validate whether all parentheses, brackets, and braces in an expression are balanced."""
    matching_pairs = {")": "(", "]": "[", "}": "{"}
    openers = set(matching_pairs.values())
    closers = set(matching_pairs.keys())

    stack = Stack(capacity=len(expression) + 1)

    for char in expression:
        if char in openers:
            stack.push(char)
        elif char in closers:
            if stack.is_empty():
                return False
            top_char = stack.pop()
            if top_char != matching_pairs[char]:
                return False

    return stack.is_empty()


# Verification
assert is_balanced_delimiters("{ [ a + (b * c) ] - d }") is True
assert is_balanced_delimiters("( [ ) ]") is False
assert is_balanced_delimiters("((())") is False
assert is_balanced_delimiters("())") is False
assert is_balanced_delimiters("") is True
print("Delimiter validator passed all test cases.")

## 4. Practical Application: Postfix (RPN) Expression Evaluation

Evaluating expressions in Reverse Polish Notation without precedence ambiguity using a Stack.


In [ ]:
def evaluate_postfix(expression: str) -> float:
    """Evaluate an arithmetic postfix expression string using a stack."""
    tokens = expression.strip().split()
    eval_stack = Stack(capacity=len(tokens) + 1)

    for token in tokens:
        if token in {"+", "-", "*", "/"}:
            if len(eval_stack) < 2:
                raise ValueError("Malformed postfix expression: insufficient operands.")
            b = eval_stack.pop()
            a = eval_stack.pop()
            if token == "+":
                res = a + b
            elif token == "-":
                res = a - b
            elif token == "*":
                res = a * b
            elif token == "/":
                if b == 0:
                    raise ZeroDivisionError("Division by zero in postfix evaluation.")
                res = a / b
            eval_stack.push(res)
        else:
            eval_stack.push(float(token))

    if len(eval_stack) != 1:
        raise ValueError("Malformed postfix expression: unconsumed operands.")
    return eval_stack.pop()


# "10 5 + 3 *" -> (10 + 5) * 3 = 45.0
assert evaluate_postfix("10 5 + 3 *") == 45.0

# "5 1 2 + 4 * + 3 -" -> 5 + ((1 + 2) * 4) - 3 = 14.0
assert evaluate_postfix("5 1 2 + 4 * + 3 -") == 14.0
print("Postfix expression evaluation verified successfully.")